# Bigbasket webscraping 

## Importing libraries

In [54]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import re
import time

## Creating as methods to scrape the data of each category

In [55]:
class WebScraping:
    y = 1000
    def fruits_vegetable_name(self,xpath):
        try:
            self.name = driver.find_element(By.XPATH,xpath).text
        except:
            self.name = 'Null'
        return self.name

    def original_price(self,xpath):
        try:
            self.og_price = driver.find_element(By.XPATH,xpath).text
        except:
            self.og_price = 'Null'
        return self.og_price

    def discount_price(self,xpath):
        try:
            self.dis_price = driver.find_element(By.XPATH,xpath).text
        except:
            self.dis_price = 'Null'
        return self.dis_price

    def brand_name(self,xpath):
        try:
            self.brand = driver.find_element(By.XPATH,xpath).text
        except:
            self.brand = 'Null'
        return self.brand

    def availability_type(self,xpath):
        try:
            self.availability = driver.find_element(By.XPATH, xpath).text
        except:
            if self.brand == 'Null':
                self.availability = 'Null'
            else:
                self.availability = 'In-stock'
        return self.availability
        

## Initiating chrome

In [56]:
driver = webdriver.Chrome()
driver.get('https://www.bigbasket.com/cl/fruits-vegetables/?nc=nb&page=1')
driver.maximize_window()

## Finding total number of items on page

In [57]:
total_items = driver.find_element(By.XPATH, '/html/body/div[2]/div[1]/div[4]/span').text
total_items = re.findall(r'\d+', total_items)
total_items = int(total_items[0])
print(total_items)

1898


## Creating empty list to store the items 

In [58]:

name_list = []
original_price_list = []
discount_price_list = []
brand_list = []
availability_list = []

In [59]:
obj = WebScraping() 

## Automatic scrolling, scraping and adding items to the lists

In [60]:
y = 0
for i in range(1, total_items+1):
    print('loading item...',i)

    # extracting the names
    xpath_name = f'/html/body/div[2]/div[1]/div[5]/div[2]/section[2]/section/ul/li[{i}]/div/div/h3/a/div/h3'


    name_list.append(obj.fruits_vegetable_name(xpath_name))
    print(name_list[i-1])

    # extracting the original price
    xpath_original_price = f'/html/body/div[2]/div[1]/div[5]/div[2]/section[2]/section/ul/li[{i}]/div/div/div[3]/div[1]/span[2]'
    original_price_list.append(obj.original_price(xpath_original_price))

    # extracting the discount price
    xpath_discount_price = f'/html/body/div[2]/div[1]/div[5]/div[2]/section[2]/section/ul/li[{i}]/div/div/div[3]/div[1]/span[1]'
    discount_price_list.append(obj.discount_price(xpath_discount_price))

    # extract brand name
    xpath_brand_name = f'/html/body/div[2]/div[1]/div[5]/div[2]/section[2]/section/ul/li[{i}]/div/div/h3/a/span'
    brand_list.append(obj.brand_name(xpath_brand_name))

    # extracting availability
    xpath_availability = f'/html/body/div[2]/div[1]/div[5]/div[2]/section[2]/section/ul/li[{i}]/div/div/div[1]/div[1]/a/div/span/span'
    availability_list.append(obj.availability_type(xpath_availability))

    try:
        driver.execute_script("window.scrollTo(0, " + str(y) + ")")
        y += 250
        time.sleep(0.1)
    except:
        pass

driver.quit()


loading item... 1
Coriander Leaves With Roots - Enhances Flavour Of The Dishes
loading item... 2
Onion (Loose)
loading item... 3
Potato
loading item... 4
Carrot - Orange (Loose)
loading item... 5
Potato
loading item... 6
Tender Coconut
loading item... 7
Coriander Leaves
loading item... 8
Mango - Banganpalli
loading item... 9
Ladies' Fingers (Loose)
loading item... 10
Carrot - Red
loading item... 11
Lemon (Loose)
loading item... 12
Cucumber
loading item... 13
Banana - Yelakki
loading item... 14
Pomegranate - Regular
loading item... 15
Palak - Cleaned, without roots
loading item... 16
Tomato - Local
loading item... 17
Tomato - Hybrid
loading item... 18
Capsicum - Green (Loose)
loading item... 19
Beans - Haricot (Loose)
loading item... 20
Lalbagh Mango - Sindhura
loading item... 21
Banana - Robusta
loading item... 22
Cucumber - English (Loose)
loading item... 23
Beetroot (Loose)
loading item... 24
Grapes - Green Sonaka
loading item... 25
Chilli - Green, Organically Grown
loading item... 2

In [61]:
print(availability_list[500:520])

['In-stock', 'In-stock', 'In-stock', 'In-stock', 'Currently unavailable', 'In-stock', 'In-stock', 'In-stock', 'In-stock', 'Currently unavailable', 'In-stock', 'Currently unavailable', 'Currently unavailable', 'Currently unavailable', 'In-stock', 'Currently unavailable', 'Currently unavailable', 'Currently unavailable', 'Currently unavailable', 'In-stock']


In [62]:
print(len(name_list))

1898


In [63]:
#item not found 
for index,item in enumerate(name_list):
    if 'Null' == item:
        print(index)

1887
1888
1889
1890
1891
1892
1893
1894
1895
1896
1897


In [64]:
import sqlite3

In [65]:
conn = sqlite3.connect('bigbasket.db')
cursor = conn.cursor()

In [66]:
cursor.execute('''CREATE TABLE IF NOT EXISTS bigbasket_data( name TEXT, original_price TEXT, discount_price TEXT, brand TEXT, availability TEXT) ''')
for i in range(len(name_list)):
    cursor.execute('INSERT INTO bigbasket_data (name, original_price, discount_price, brand, availability) VALUES (?, ?, ?, ?, ?)',(name_list[i], original_price_list[i], discount_price_list[i], brand_list[i], availability_list[i]))
conn.commit()
conn.close()